# Sentence-level tactic attribution

The classifier returns one label for a whole ad. On a 22-word ad that is fine.
On a 588-word video transcript it is close to useless — the ad uses several
tactics in different places, and a single label hides all of it.

This runs the model **per sentence**, so the panel can say *this sentence is a
fear appeal, that one borrows authority*, and highlight each.

Why not anchor on trigger phrases instead: `find_phrases()` already returns
exact spans, but the lexicon only knows the 87 patterns written into it. On the
Cheers transcript almost none of them fire — the persuasion there is enzyme
names and age-decline framing, which no regex covers. Sentence classification
finds those; the lexicon is then overlaid as extra evidence where it applies.

**Caveat worth stating in the report:** the model was fine-tuned on whole ads,
not sentences, so per-sentence inference is mildly out of distribution. A
sentence also loses its context. Treat the output as attribution, not as an
independently reliable classification.

## Setup

In [1]:
import re, sys
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "app" / "tactics.py").exists():
    if REPO.parent == REPO:
        raise FileNotFoundError("run this notebook from anywhere inside the Group-6-Final-Project repo")
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "app"))

import predict, tactics

print("model ready:", predict.is_ready())
print("guard threshold:", tactics.LOW_CONFIDENCE)

model ready: True
guard threshold: 0.6


## Cleaning and splitting

Transcript timestamps and generator footers are stripped first — they are not ad
copy and would be classified as if they were. Offsets are kept so the front end
can highlight the original text.

In [2]:
TIMESTAMP = re.compile(r"^\s*\d{1,2}:\d{2}:\d{2}\s*$", re.M)
FOOTER    = re.compile(r"^\s*>\s*Generated by.*$", re.M | re.I)
UI_NOISE  = re.compile(r"\b(?:Follow|Add comment|Show more|Like|Share|Sponsored)\b", re.I)

def clean_transcript(text):
    text = FOOTER.sub("", text)
    text = TIMESTAMP.sub(" ", text)
    text = UI_NOISE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

SENT_END = re.compile(r'(?<=[.!?])\s+(?=[A-Z"\'(])')

def split_sentences(text, min_words=4):
    """Split into sentences, keeping (start, end) offsets into `text`.

    Fragments shorter than min_words are merged into the previous sentence —
    'Oh, look at that.' carries no tactic on its own and classifying it alone
    produces noise.
    """
    spans, start = [], 0
    for m in SENT_END.finditer(text):
        seg = text[start:m.start()].strip()
        if seg:
            spans.append((start, m.start(), seg))
        start = m.end()
    tail = text[start:].strip()
    if tail:
        spans.append((start, len(text), tail))

    merged = []
    for s, e, t in spans:
        if merged and len(t.split()) < min_words:
            ps, _, pt = merged[-1]
            merged[-1] = (ps, e, (pt + " " + t).strip())
        else:
            merged.append((s, e, t))
    return merged

print("ready")

ready


## Load an ad

In [3]:
SOURCE = REPO / "modeling" / "data" / "cheers_ad_transcript.txt"
if not SOURCE.exists():
    raise FileNotFoundError(
        f"{SOURCE} not found — the Cheers Protect video transcript is third-party ad "
        "content and is not committed. Drop the transcript text file at that path to rerun "
        "(see modeling/data/README.md).")

raw = Path(SOURCE).read_text(encoding="utf-8")
body = clean_transcript(raw)
sentences = split_sentences(body)

print(f"{len(body.split())} words -> {len(sentences)} sentences")
lens = [len(t.split()) for _, _, t in sentences]
print(f"words per sentence: min {min(lens)} median {sorted(lens)[len(lens)//2]} max {max(lens)}")

584 words -> 33 sentences
words per sentence: min 5 median 14 max 48


## Classify each sentence

`THRESHOLD` decides what counts as "this sentence uses a tactic". Start at the
project's own guard value and tune it by reading the output below — too low and
every sentence gets a label, too high and the panel stays empty.

In [4]:
THRESHOLD = tactics.LOW_CONFIDENCE   # 0.60 to start
NEUTRAL   = "Neutral"

results = []
for start, end, sent in sentences:
    try:
        p = predict.predict(sent)
    except Exception as e:
        continue
    phrases = tactics.find_phrases(sent)
    results.append({
        "start": start, "end": end, "text": sent,
        "label": p["label"], "confidence": p["confidence"],
        "distribution": p["distribution"],
        "phrases": {k: v for k, v in phrases.items() if v},
    })

print(f"classified {len(results)} sentences\n")
for r in results:
    mark = "*" if r["confidence"] >= THRESHOLD and r["label"] != NEUTRAL else " "
    ph = f"  [{', '.join(r['phrases'])}]" if r["phrases"] else ""
    print(f"{mark} {r['label']:<24}{r['confidence']:>6.0%}  {r['text'][:62]}{ph}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

classified 33 sentences

* Scarcity                   69%  And let's see what happens when we pour in the acetaldehyde.
* Scarcity                   76%  Oh, look at that. It's not magic.
* Authority Manipulation     84%  It looks it, but it's science.
* Authority Manipulation     83%  Everyone always says that alcohol is toxic, and they are wrong
  Scarcity                   43%  So when you drink alcohol, alcohol is really not actually that
* Fear Appeals               70%  It becomes twenty times more toxic when in your liver, there's
* Fear Appeals               81%  And acetaldehyde is twenty times more toxic than the alcohol i
* Social Proof               83%  Now, typically, your liver uses an enzyme called ALDH to conve
  Fear Appeals               56%  And the problem is, is that your liver can only process about 
* Scarcity                   76%  And if you're drinking more than one standard drink per hour, 
* Fear Appeals               92%  You're gonna have a toxic buildup 

## Merge adjacent sentences

Consecutive sentences with the same tactic are one passage, not several
findings. Merging keeps the panel readable and the highlight contiguous.

In [5]:
def merge_runs(results, threshold, neutral=NEUTRAL):
    blocks = []
    for r in results:
        if r["label"] == neutral or r["confidence"] < threshold:
            continue
        if blocks and blocks[-1]["label"] == r["label"] and r["start"] - blocks[-1]["end"] < 3:
            b = blocks[-1]
            b["end"] = r["end"]
            b["text"] = (b["text"] + " " + r["text"]).strip()
            b["confidence"] = max(b["confidence"], r["confidence"])
            b["sentences"] += 1
            for k, v in r["phrases"].items():
                b["phrases"].setdefault(k, []).extend(v)
        else:
            blocks.append({**r, "sentences": 1, "phrases": dict(r["phrases"])})
    return blocks

blocks = merge_runs(results, THRESHOLD)

print(f"{len(blocks)} tactic passages found\n")
for b in blocks:
    print(f"--- {tactics.display_name(b['label'])}  ({b['confidence']:.0%}, "
          f"{b['sentences']} sentence(s), chars {b['start']}-{b['end']}) ---")
    print(f"    \"{b['text'][:150]}{'...' if len(b['text'])>150 else ''}\"")
    ph = [p["text"] for lst in b["phrases"].values() for p in lst]
    if ph:
        print(f"    trigger phrases: {ph}")
    print()

15 tactic passages found

--- Scarcity  (76%, 2 sentence(s), chars 0-94) ---
    "And let's see what happens when we pour in the acetaldehyde. Oh, look at that. It's not magic."

--- Authority  (84%, 2 sentence(s), chars 95-267) ---
    "It looks it, but it's science. Everyone always says that alcohol is toxic, and they are wrong, and I'm gonna explain why with science and what you can..."

--- Fear  (81%, 2 sentence(s), chars 338-543) ---
    "It becomes twenty times more toxic when in your liver, there's an enzyme called ADH, which converts it into acetaldehyde. And acetaldehyde is twenty t..."

--- Social proof  (83%, 1 sentence(s), chars 544-640) ---
    "Now, typically, your liver uses an enzyme called ALDH to convert this acetaldehyde into acetate."

--- Scarcity  (76%, 1 sentence(s), chars 910-988) ---
    "And if you're drinking more than one standard drink per hour, then guess what?"

--- Fear  (99%, 2 sentence(s), chars 989-1291) ---
    "You're gonna have a toxic buildup of 

## The panel this produces

In [ ]:
def render_panel(blocks):
    if not blocks:
        print("Nothing pushy found.")
        print("This ad reads as straightforward. You can still take your time.")
        return

    by_label = {}
    for b in blocks:
        by_label.setdefault(b["label"], []).append(b)

    n = len(by_label)
    noun = "pressure tactic" if n == 1 else "pressure tactics"
    print(f"Take your time.")
    print(f"This ad uses {n} {noun}. Nothing requires a decision today.\n")

    for label, bs in sorted(by_label.items(), key=lambda kv: -max(b["confidence"] for b in kv[1])):
        best = max(bs, key=lambda b: b["confidence"])
        phrases = [p for b in bs for lst in b["phrases"].values() for p in lst]
        print(f"{tactics.display_name(label).upper()}")
        print(f"  {tactics.explain(label, phrases)}")
        for b in bs:
            snippet = b["text"][:120] + ("..." if len(b["text"]) > 120 else "")
            print(f'  \u2192 "{snippet}"')
        print()

render_panel(blocks)

## Tune the threshold

Sweep it and look at what appears and disappears. The right value shows the
tactics a reader would agree are there, without labelling ordinary sentences.

In [ ]:
for t in [0.40, 0.50, 0.60, 0.70, 0.80]:
    bs = merge_runs(results, t)
    labels = sorted({b["label"] for b in bs})
    print(f"threshold {t:.2f} -> {len(bs)} passages, tactics: {labels or 'none'}")

## Compare against whole-ad classification

The point of the exercise: what does the single-label verdict miss?

In [ ]:
whole = predict.predict(body)
print("WHOLE AD")
print(f"  {whole['label']} at {whole['confidence']:.0%}"
      f"   ({whole.get('windows', 1)} window(s), {whole.get('token_count', '?')} subwords)")
print("  distribution:", ", ".join(f"{d['label']} {d['confidence']:.0%}"
                                   for d in whole["distribution"][:4]))

print("\nPER SENTENCE")
found = sorted({b["label"] for b in blocks})
print(f"  tactics located: {found}")
print(f"  passages: {len(blocks)}")

missed = set(found) - {whole["label"]}
if missed:
    print(f"\n  The whole-ad verdict misses: {sorted(missed)}")
    print("  Each of those is a tactic the model itself found, in a specific")
    print("  passage, that single-label classification collapsed away.")

## Notes

- Sentence-level inference is out of distribution for a model fine-tuned on
  whole ads. Report it as attribution, not as independent classification.
- A sentence stripped of context can read differently than it does in place.
- This is also the clearest argument for a multi-label head: the model is
  already finding several tactics in one ad, and softmax is discarding all but
  one of them.
- Offsets are preserved end to end, so `app.js` and `content.js` can highlight
  the original text directly.